In [ ]:
import pandas as pd
import numpy as np
import time
import multiprocessing as mp

FILE_PATH = r'C:\Users\lipat\Desktop\CODES\School\PDC\MidtermProjectParallelDataProcessingSystem\data\healthy_foods_database.csv'

def run_sequential(df):
    start = time.perf_counter()
    # Ops: Filter (col 1), Aggregate (col 2), Sort
    # Using .iloc ensures we don't need to know the column names
    filtered = df[df.iloc[:, 1].notnull()] 
    summary = filtered.groupby(filtered.columns[0])[filtered.columns[1]].mean()
    result = summary.sort_values(ascending=False)
    return time.perf_counter() - start

def process_chunk(chunk):
    filtered = chunk[chunk.iloc[:, 1].notnull()]
    # Returns sum and count of the second column grouped by the first column
    return filtered.groupby(filtered.columns[0])[filtered.columns[1]].agg(['sum', 'count'])

def run_parallel(df):
    cores = mp.cpu_count()
    chunks = np.array_split(df, cores)
    start = time.perf_counter()
    with mp.Pool(cores) as pool:
        results = pool.map(process_chunk, chunks)
    combined = pd.concat(results).groupby(level=0).sum()
    final_mean = combined['sum'] / combined['count']
    return time.perf_counter() - start

if __name__ == '__main__':
    data = pd.read_csv(FILE_PATH)
    # Ensure the second column is numeric for math
    data.iloc[:, 1] = pd.to_numeric(data.iloc[:, 1], errors='coerce')
    
    print(f"Processing {len(data)} rows...")
    s_time = run_sequential(data)
    p_time = run_parallel(data)
    
    print(f"Sequential: {s_time:.4f}s | Parallel: {p_time:.4f}s")
    print(f"Speedup: {s_time / p_time:.2f}x")

Processing 9028 rows...


c:\Users\lipat\anaconda3\Lib\site-packages\numpy\_core\fromnumeric.py:57: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)
